# **Eksperimen UDAO-LASFNet pada Dataset DroneVehicle (RGB-IR)**
Notebook ini menjalankan pipeline eksperimen **UDAO-LASFNet** (*Uncertainty-Driven Deformable Alignment & Differential Feature Attention Transformation*) dari repositori `Allezan/UDAO-LASFNet` pada dataset **DroneVehicle (5 Kelas: `car`, `truck`, `bus`, `van`, `freight car`)**:
1. **UDAO-ASFF** (Layer 10, $P_3/8$): Penyelarasan spasial lintas-modalitas berbasis DCNv2 dengan *Cross-Modal Pointwise-Depthwise Offset Generator* dan *Static-Deformable Dual-Path Residual*.
2. **Multi-Scale D-FATM** (Layer 24, 29, 33 pada Neck $P_3/8, P_4/16, P_5/32$): Peningkatan atensi batas kontur kendaraan implisit (*Sobel + 8-neighbor Laplacian* + *Residual Boundary-Guided Attention*) pada seluruh skala deteksi dengan *Decoupled Shape-IoU Regression*.

### **LANGKAH 1: Inisialisasi Environment, Google Drive, & Clone Repositori `Allezan/UDAO-LASFNet`**

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/Skripsi (Tugas Akhir)/Laboratory/UDAO-LASFNet/DroneVehicle"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Folder target di Google Drive siap: {SAVE_DIR}")

%cd /content
!rm -rf /content/UDAO-LASFNet
!git clone -b udao-dfatm https://github.com/Allezan/UDAO-LASFNet.git
%cd /content/UDAO-LASFNet

!pip install -q "scikit-image>=0.24.0" xlsxwriter huggingface_hub thop pycocotools ultralytics albumentations timm einops prettytable gitpython
print("Instalasi dependensi & clone repositori Allezan/UDAO-LASFNet selesai!")

### **LANGKAH 2: Unduh Dataset McCheng/DroneVehicle dari Hugging Face**

In [ ]:
import os
import subprocess
from huggingface_hub import snapshot_download

target_dir = "/content/UDAO-LASFNet/dataset/dronevehicle_new"
os.makedirs(target_dir, exist_ok=True)

print("Mengunduh dataset dari Hugging Face...")
snapshot_download(repo_id="McCheng/DroneVehicle", repo_type="dataset", local_dir=target_dir)

for zip_name in ['train.zip', 'val.zip', 'test.zip']:
    zip_path = os.path.join(target_dir, zip_name)
    if os.path.exists(zip_path):
        print(f"Mengekstrak {zip_name}...")
        subprocess.run(["unzip", "-q", "-o", zip_path, "-d", target_dir])

print("Ekstraksi dataset selesai!")

### **LANGKAH 3: Penataan Struktur Direktori & Pembersihan File Bermasalah**

In [ ]:
import os

target_dir = "/content/UDAO-LASFNet/dataset/dronevehicle_new"

for mode in ['train', 'val', 'test']:
    os.makedirs(f"{target_dir}/data_rgb/{mode}/labels", exist_ok=True)
    os.makedirs(f"{target_dir}/data_ir/{mode}/labels", exist_ok=True)
    os.makedirs(f"{target_dir}/data_rgb/{mode}/images", exist_ok=True)
    os.makedirs(f"{target_dir}/data_ir/{mode}/images", exist_ok=True)

print("Menata letak file gambar dan label...")
for mode in ['train', 'val', 'test']:
    !mv {target_dir}/{mode}/{mode}img/* {target_dir}/data_rgb/{mode}/images/ 2>/dev/null || true
    !mv {target_dir}/{mode}/{mode}imgr/* {target_dir}/data_ir/{mode}/images/ 2>/dev/null || true
    !mv {target_dir}/{mode}/{mode}label/* {target_dir}/data_rgb/{mode}/labels/ 2>/dev/null || true
    !mv {target_dir}/{mode}/{mode}labelr/* {target_dir}/data_ir/{mode}/labels/ 2>/dev/null || true

problematic_files = ["04895", "06463", "14048", "12900", "05001", "09012", "04991", "16009", "14035"]

print("\n--- Menghapus File Bermasalah secara Manual ---")
for base_name in problematic_files:
    for mode in ['train', 'val', 'test']:
        paths_to_remove = [
            f"data_rgb/{mode}/images/{base_name}.jpg",
            f"data_ir/{mode}/images/{base_name}.jpg",
            f"data_rgb/{mode}/labels/{base_name}.txt",
            f"data_ir/{mode}/labels/{base_name}.txt",
            f"data_rgb/{mode}/labels/{base_name}.xml",
            f"data_ir/{mode}/labels/{base_name}.xml"
        ]
        for p in paths_to_remove:
            full_path = os.path.join(target_dir, p)
            if os.path.exists(full_path):
                os.remove(full_path)
                print(f"Dihapus: {p}")

print("Pembersihan file manual selesai!")

### **LANGKAH 4: Pemotongan Border Putih 100px (Crop $640 \times 512$) & Konversi XML ke YOLO TXT Berbasis Shape-Aware**
Tahap ini melakukan:
1. **Pemotongan Citra ($640 \times 512$):** Membuang border putih 100 piksel (`img[100:612, 100:740]`) pada seluruh citra RGB dan IR untuk mengembalikan resolusi asli kendaraan drone 100% dan menghilangkan false-edge gradien.
2. **Transformasi Koordinat Bounding Box:** Menggeser koordinat ($x_{\text{baru}} = x - 100$, $y_{\text{baru}} = y - 100$), melakukan boundary clipping, dan normalisasi terhadap dimensi $640 \times 512$.
3. **Sinkronisasi Multimodal & Kelengkapan Anotasi Malam Hari:** Memastikan pasangan RGB-IR-Label tersinkronisasi sempurna serta melengkapi anotasi objek malam hari dari kanal IR (`labelr`) ketika anotasi RGB (`label`) kosong/kurang akibat kondisi gelap.

In [ ]:
import os
import glob
import shutil
import cv2
import xml.etree.ElementTree as ET
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

target_dir = "/content/UDAO-LASFNet/dataset/dronevehicle_new"

# 1. Pemotongan Citra 100px Border Putih (840x712 -> 640x512)
def crop_single_image(img_path):
    img = cv2.imread(img_path)
    if img is None:
        return
    h, w = img.shape[:2]
    # Hanya crop jika masih berukuran 840x712
    if w == 840 and h == 712:
        cropped = img[100:612, 100:740]  # height: 512, width: 640
        cv2.imwrite(img_path, cropped)

def crop_all_images(base_dir):
    print("--- [1/3] Memotong Border Putih 100px pada Citra RGB & IR ---")
    all_imgs = []
    for mode in ['train', 'val', 'test']:
        for mod in ['data_rgb', 'data_ir']:
            p = os.path.join(base_dir, mod, mode, "images", "*.*")
            all_imgs.extend([f for f in glob.glob(p) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

    print(f"Total citra yang diproses: {len(all_imgs)} citra")
    with ThreadPoolExecutor(max_workers=8) as executor:
        list(tqdm(executor.map(crop_single_image, all_imgs), total=len(all_imgs), desc="Memotong border 100px"))
    print("✅ Pemotongan seluruh citra ke 640x512 selesai!")

crop_all_images(target_dir)

# 2. Konversi XML ke Format TXT YOLO dengan Pergeseran Koordinat (-100px)
def convert_dronevehicle_xml_to_yolo_cropped(xml_dir, txt_dir, width=640.0, height=512.0):
    if not os.path.exists(xml_dir):
        return
    os.makedirs(txt_dir, exist_ok=True)
    class_map = {
        'car': 0,
        'truck': 1,
        'bus': 2,
        'van': 3,
        'freight car': 4,
        'freight_car': 4,
        'feright car': 4,
        'feright_car': 4,
        'feright': 4
    }
    xml_files = glob.glob(os.path.join(xml_dir, "*.xml"))

    for xml_file in xml_files:
        tree = ET.parse(xml_file)
        root = tree.getroot()
        filename = os.path.splitext(os.path.basename(xml_file))[0]
        txt_file_path = os.path.join(txt_dir, f"{filename}.txt")

        yolo_lines = []
        for obj in root.findall('object'):
            name = obj.find('name').text.strip().lower()
            if name not in class_map:
                continue
            class_id = class_map[name]
            polygon = obj.find('polygon')
            bndbox = obj.find('bndbox')

            if polygon is not None:
                try:
                    xs = [float(polygon.find(f'x{i}').text) for i in range(1, 5)]
                    ys = [float(polygon.find(f'y{i}').text) for i in range(1, 5)]
                    raw_xmin, raw_xmax = min(xs), max(xs)
                    raw_ymin, raw_ymax = min(ys), max(ys)
                except:
                    continue
            elif bndbox is not None:
                try:
                    raw_xmin = float(bndbox.find('xmin').text)
                    raw_ymin = float(bndbox.find('ymin').text)
                    raw_xmax = float(bndbox.find('xmax').text)
                    raw_ymax = float(bndbox.find('ymax').text)
                except:
                    continue
            else:
                continue

            # Pergeseran koordinat akibat crop 100px
            xmin = max(0.0, min(width, raw_xmin - 100.0))
            xmax = max(0.0, min(width, raw_xmax - 100.0))
            ymin = max(0.0, min(height, raw_ymin - 100.0))
            ymax = max(0.0, min(height, raw_ymax - 100.0))

            box_w = xmax - xmin
            box_h = ymax - ymin

            # Abaikan objek yang terpotong habis di luar batas (< 3 px)
            if box_w < 3.0 or box_h < 3.0:
                continue

            x_center = (xmin + xmax) / 2.0 / width
            y_center = (ymin + ymax) / 2.0 / height
            norm_w = box_w / width
            norm_h = box_h / height

            yolo_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}")

        with open(txt_file_path, "w") as f:
            f.write("\n".join(yolo_lines))

print("\n--- [2/3] Mengonversi XML ke Format TXT YOLO Berkoordinat Baru ---")
for mode in ['train', 'val', 'test']:
    convert_dronevehicle_xml_to_yolo_cropped(f"{target_dir}/data_rgb/{mode}/labels", f"{target_dir}/data_rgb/{mode}/labels")
    convert_dronevehicle_xml_to_yolo_cropped(f"{target_dir}/data_ir/{mode}/labels", f"{target_dir}/data_ir/{mode}/labels")
    !find {target_dir}/data_rgb/{mode}/labels -name "*.xml" -delete 2>/dev/null || true
    !find {target_dir}/data_ir/{mode}/labels -name "*.xml" -delete 2>/dev/null || true

# 3. Sinkronisasi Pasangan RGB-IR-Label & Penyempurnaan Anotasi Malam Hari
def synchronize_dataset(base_dir):
    print("\n--- [3/3] Sinkronisasi Pasangan Dataset Multimodal ---")
    for mode in ['train', 'val', 'test']:
        rgb_img_dir = os.path.join(base_dir, f"data_rgb/{mode}/images")
        rgb_lbl_dir = os.path.join(base_dir, f"data_rgb/{mode}/labels")
        ir_img_dir = os.path.join(base_dir, f"data_ir/{mode}/images")
        ir_lbl_dir = os.path.join(base_dir, f"data_ir/{mode}/labels")

        rgb_imgs = {os.path.splitext(f)[0] for f in os.listdir(rgb_img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))} if os.path.exists(rgb_img_dir) else set()
        rgb_lbls = {os.path.splitext(f)[0] for f in os.listdir(rgb_lbl_dir) if f.lower().endswith('.txt')} if os.path.exists(rgb_lbl_dir) else set()
        ir_imgs = {os.path.splitext(f)[0] for f in os.listdir(ir_img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))} if os.path.exists(ir_img_dir) else set()
        ir_lbls = {os.path.splitext(f)[0] for f in os.listdir(ir_lbl_dir) if f.lower().endswith('.txt')} if os.path.exists(ir_lbl_dir) else set()

        valid_keys = rgb_imgs.intersection(rgb_lbls).intersection(ir_imgs).intersection(ir_lbls)

        # Sinkronisasi kelengkapan anotasi objek malam hari (IR -> RGB jika RGB kosong/kurang pada citra gelap)
        synced_night_labels = 0
        for k in valid_keys:
            p_rgb = os.path.join(rgb_lbl_dir, f"{k}.txt")
            p_ir = os.path.join(ir_lbl_dir, f"{k}.txt")
            with open(p_rgb, "r") as f1, open(p_ir, "r") as f2:
                lines_rgb = [line for line in f1.read().strip().splitlines() if line.strip()]
                lines_ir = [line for line in f2.read().strip().splitlines() if line.strip()]
            if len(lines_ir) > len(lines_rgb):
                shutil.copyfile(p_ir, p_rgb)
                synced_night_labels += 1
            elif len(lines_rgb) > len(lines_ir):
                shutil.copyfile(p_rgb, p_ir)

        print(f"Jumlah sampel sinkron {mode.upper()}: {len(valid_keys)} (Anotasi malam IR->RGB dilengkapi: {synced_night_labels} sampel)")

        def clean_unpaired(directory, valid_set, exts):
            if not os.path.exists(directory): return
            for f in os.listdir(directory):
                name, ext = os.path.splitext(f)
                if ext.lower() in exts and name not in valid_set:
                    try: os.remove(os.path.join(directory, f))
                    except: pass

        clean_unpaired(rgb_img_dir, valid_keys, ['.jpg', '.jpeg', '.png'])
        clean_unpaired(rgb_lbl_dir, valid_keys, ['.txt'])
        clean_unpaired(ir_img_dir, valid_keys, ['.jpg', '.jpeg', '.png'])
        clean_unpaired(ir_lbl_dir, valid_keys, ['.txt'])

synchronize_dataset(target_dir)
!find {target_dir} -name "*.cache" -delete 2>/dev/null || true
print("\n✅ Dataset DroneVehicle $640 \times 512$ Sukses Dipotong, Dikonversi, dan Bersih!")

### **LANGKAH 4B: Verifikasi Visual Bounding Box pada Citra Hasil Crop (Sanity Check)**
Sel ini memuat 3 sampel acak dari dataset yang baru saja dipotong dan menggambar bounding box hasil pergeseran koordinat.
Pastikan visualisasi menunjukkan bahwa:
1. **Border putih 100px sudah hilang total** (gambar murni $640 \times 512$).
2. **Kotak bounding box menempel 100% presisi di bodi kendaraan**.

In [ ]:
import os
import random
import cv2
import matplotlib.pyplot as plt

target_dir = "/content/UDAO-LASFNet/dataset/dronevehicle_new"
classes = ['car', 'truck', 'bus', 'van', 'freight car']
colors = [(0, 255, 0), (255, 0, 0), (0, 0, 255), (255, 255, 0), (255, 0, 255)]

rgb_dir = f"{target_dir}/data_rgb/train/images"
rgb_lbl = f"{target_dir}/data_rgb/train/labels"
ir_dir = f"{target_dir}/data_ir/train/images"

sample_names = [os.path.splitext(f)[0] for f in os.listdir(rgb_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
selected = random.sample(sample_names, min(3, len(sample_names)))

fig, axes = plt.subplots(len(selected), 2, figsize=(14, 5 * len(selected)))
if len(selected) == 1:
    axes = [axes]

for i, name in enumerate(selected):
    rgb_path = os.path.join(rgb_dir, f"{name}.jpg")
    ir_path = os.path.join(ir_dir, f"{name}.jpg")
    lbl_path = os.path.join(rgb_lbl, f"{name}.txt")

    img_rgb = cv2.imread(rgb_path)
    img_ir = cv2.imread(ir_path)
    if img_rgb is None or img_ir is None:
        continue
    h, w = img_rgb.shape[:2]

    # Baca label
    if os.path.exists(lbl_path):
        with open(lbl_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls_id = int(parts[0])
                    xc, yc, bw, bh = map(float, parts[1:5])
                    x1 = int((xc - bw / 2) * w)
                    y1 = int((yc - bh / 2) * h)
                    x2 = int((xc + bw / 2) * w)
                    y2 = int((yc + bh / 2) * h)
                    color = colors[cls_id % len(colors)]
                    label_name = classes[cls_id] if cls_id < len(classes) else str(cls_id)
                    cv2.rectangle(img_rgb, (x1, y1), (x2, y2), color, 2)
                    cv2.putText(img_rgb, label_name, (x1, max(y1 - 5, 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)
                    cv2.rectangle(img_ir, (x1, y1), (x2, y2), color, 2)
                    cv2.putText(img_ir, label_name, (x1, max(y1 - 5, 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)

    axes[i][0].imshow(cv2.cvtColor(img_rgb, cv2.COLOR_BGR2RGB))
    axes[i][0].set_title(f"RGB Cropped ({w}x{h}) - Sample: {name}")
    axes[i][0].axis('off')

    axes[i][1].imshow(cv2.cvtColor(img_ir, cv2.COLOR_BGR2RGB))
    axes[i][1].set_title(f"IR Cropped ({w}x{h}) - Sample: {name}")
    axes[i][1].axis('off')

plt.tight_layout()
plt.show()
print("✅ Verifikasi visual selesai! Jika kotak presisi menempel di kendaraan, lanjutkan ke Langkah 5.")

In [ ]:
import zipfile
import cv2
import numpy as np
import matplotlib.pyplot as plt

sample_id = "04502"  # Contoh sampel bus
zip_path = "/content/UDAO-LASFNet/dataset/dronevehicle_new/train.zip"

# 1. Baca citra ASLI dari dalam zip (BEFORE: 840x712)
with zipfile.ZipFile(zip_path, 'r') as z:
    img_bytes = z.read(f"train/trainimg/{sample_id}.jpg")
    img_before = cv2.imdecode(np.frombuffer(img_bytes, np.uint8), cv2.IMREAD_COLOR)

# 2. Baca citra HASIL CROP (AFTER: 640x512)
img_after = cv2.imread(f"/content/UDAO-LASFNet/dataset/dronevehicle_new/data_rgb/train/images/{sample_id}.jpg")

# 3. Plot Perbandingan Berdampingan
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Before
axes[0].imshow(cv2.cvtColor(img_before, cv2.COLOR_BGR2RGB))
axes[0].set_title(f"BEFORE: Citra Mentah ({img_before.shape[1]}x{img_before.shape[0]})\nPerhatikan Bingkai Putih Tebal 100px di Sekelilingnya!", fontsize=12, color='red')
axes[0].axis('on')  # Skala koordinat piksel dinyalakan agar terlihat jelas

# After
axes[1].imshow(cv2.cvtColor(img_after, cv2.COLOR_BGR2RGB))
axes[1].set_title(f"AFTER: Citra Hasil Crop ({img_after.shape[1]}x{img_after.shape[0]})\nBingkai Putih Hilang Total, 100% Scene Drone Aktif!", fontsize=12, color='green')
axes[1].axis('on')

plt.tight_layout()
plt.show()

### **LANGKAH 5: Konfigurasi File YAML Dataset (`data/dronevehicle_new.yaml`) & Patch Kompatibilitas PyTorch 2.6+**

In [ ]:
import os

REPO_DIR = "/content/UDAO-LASFNet"

yaml_content = """# Dataset DroneVehicle untuk UDAO-LASFNet
path: /content/UDAO-LASFNet/dataset/dronevehicle_new
train_rgb: data_rgb/train/images
train_ir: data_ir/train/images
val_rgb: data_rgb/val/images
val_ir: data_ir/val/images
test_rgb: data_rgb/test/images
test_ir: data_ir/test/images

# Classes
nc: 5
names:
  0: car
  1: truck
  2: bus
  3: van
  4: freight car
"""

os.makedirs(f"{REPO_DIR}/data", exist_ok=True)
yaml_path = f"{REPO_DIR}/data/dronevehicle_new.yaml"
with open(yaml_path, "w") as f:
    f.write(yaml_content)
print(f"File konfigurasi YAML sukses dibuat di: {yaml_path}")

# Patch kompatibilitas PyTorch 2.6+ (weights_only=False) pada train.py dan val.py
header_patch = '''import sys
import torch
import utils.general as ug
ug.check_requirements = lambda *args, **kwargs: True
_orig_load = torch.load
def _patched_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _orig_load(*args, **kwargs)
torch.load = _patched_load
'''
for fname in ["train.py", "val.py"]:
    fpath = os.path.join(REPO_DIR, fname)
    with open(fpath, "r") as f:
        code = f.read()
    if "_patched_load" not in code:
        with open(fpath, "w") as f:
            f.write(header_patch + "\n" + code)
print("Patch kompatibilitas PyTorch 2.6+ siap!")

### **LANGKAH 6: Sanity Check & Profiling Arsitektur UDAO-LASFNet (< 8.0M Parameters)**

In [ ]:
%cd /content/UDAO-LASFNet
import torch
from models.yolo import Model

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
model = Model('models/udao_lasfnet.yaml', ch=3, nc=5).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal Parameters (Unfused): {total_params:,} ({total_params/1e6:.3f}M)")
assert total_params < 8_000_000, "Parameter melebihi batas 8.0M!"
print("Sanity Check LULUS: Model berada di bawah batas 8.0M parameter!")

### **LANGKAH 7: Mulai Proses Training UDAO-LASFNet (300 Epochs)**
> **TIPS PENTING**:
> - Jika di Google Drive sudah ada folder lama `Full_UDAO_LASFNet_Cropped_ShapeIoU`, kamu disarankan untuk me-rename folder lama tersebut terlebih dahulu agar hasil training baru dari Epoch 0 ini tersimpan bersih dan tidak tercampur.
> - Sel ini otomatis mengaktifkan `--cos-lr`, `--image-weights`, dan `--save-period 20` sehingga checkpoint tersimpan tiap 20 epoch.

In [ ]:
%cd /content/UDAO-LASFNet

!python train.py \
  --weights "" \
  --cfg models/udao_lasfnet.yaml \
  --data data/dronevehicle_new.yaml \
  --hyp data/hyps/hyp.scratch.yaml \
  --epochs 300 \
  --batch-size 48 \
  --imgsz 640 \
  --device 0 \
  --workers 8 \
  --cos-lr \
  --image-weights \
  --project "/content/drive/MyDrive/Skripsi (Tugas Akhir)/Laboratory/UDAO-LASFNet/DroneVehicle" \
  --name "Full_UDAO_LASFNet_Cropped_ShapeIoU" \
  --save-period 20 \
  --exist-ok

# JIKA TERPUTUS DI TENGAH JALAN, KOMENTARI PERINTAH DI ATAS DAN JALANKAN RESUME DI BAWAH INI:
# !python train.py --resume "/content/drive/MyDrive/Skripsi (Tugas Akhir)/Laboratory/UDAO-LASFNet/DroneVehicle/Full_UDAO_LASFNet_Cropped_ShapeIoU/weights/last.pt"

### **LANGKAH 8: Evaluasi Model Terbaik (`best.pt`) & Visualisasi Kurva**

In [ ]:
%cd /content/UDAO-LASFNet

# 1. Evaluasi akurasi secara mendalam (Tabel Per-Kelas Penuh)
!python val.py \
  --data data/dronevehicle_new.yaml \
  --weights "/content/drive/MyDrive/Skripsi (Tugas Akhir)/Laboratory/UDAO-LASFNet/DroneVehicle/Full_UDAO_LASFNet_Cropped_ShapeIoU/weights/best.pt" \
  --batch-size 32 \
  --imgsz 640 \
  --device 0 \
  --task val \
  --verbose

# 2. Visualisasi Kurva mAP & Loss Training
import pandas as pd
import matplotlib.pyplot as plt

csv_path = "/content/drive/MyDrive/Skripsi (Tugas Akhir)/Laboratory/UDAO-LASFNet/DroneVehicle/Full_UDAO_LASFNet_Cropped_ShapeIoU/results.csv"
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Kurva mAP
    if 'metrics/mAP_0.5' in df.columns:
        axes[0].plot(df['epoch'], df['metrics/mAP_0.5'] * 100, label='mAP@0.5 (Shape-IoU + Cropped)', color='blue', lw=2)
    if 'metrics/mAP_0.5:0.95' in df.columns:
        axes[0].plot(df['epoch'], df['metrics/mAP_0.5:0.95'] * 100, label='mAP@0.5:0.95', color='green', lw=1.5, ls='--')
    axes[0].axhline(y=85.7, color='red', linestyle=':', label='Target Paper LASFNet (85.7%)', lw=2)
    axes[0].set_title('Evaluasi mAP UDAO-LASFNet pada DroneVehicle (Cropped)')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('mAP (%)')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Kurva Total Loss
    if 'train/box_loss' in df.columns and 'val/box_loss' in df.columns:
        axes[1].plot(df['epoch'], df['train/box_loss'], label='Train Box Loss (Shape-IoU)', color='orange')
        axes[1].plot(df['epoch'], df['val/box_loss'], label='Val Box Loss', color='purple', ls='--')
    axes[1].set_title('Konvergensi Loss Bounding Box')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print(f"File {csv_path} belum ditemukan. Jalankan training terlebih dahulu!")

---
### **LANGKAH 9 (OPSIONAL - ABLATION STUDY): Uji Kontribusi Tiap Modul Secara Terpisah**
Bagian ini digunakan untuk mengisi **Tabel Ablation Study di Bab 4 Skripsi**:
1. **Ablation Study 1 (`UDAO-Only`)**: Menggunakan `models/lasfnet_udao_only.yaml` (`UDAO_ASFF` di Layer 10 + `LCBHAM` standar di Layer 24).
2. **Ablation Study 2 (`DFATM-Only`)**: Menggunakan `models/lasfnet_dfatm_only.yaml` (`MFS` standar di Layer 10 + Multi-Scale `D_FATM` di Layer 24, 29, 33).

In [ ]:
# === ABLATION STUDY 1: UDAO-ONLY (Training + Evaluasi) ===
%cd /content/UDAO-LASFNet

!python train.py \
  --weights "" \
  --cfg models/lasfnet_udao_only.yaml \
  --data data/dronevehicle_new.yaml \
  --hyp data/hyps/hyp.scratch.yaml \
  --epochs 300 \
  --batch-size 48 \
  --imgsz 640 \
  --device 0 \
  --workers 8 \
  --cos-lr \
  --image-weights \
  --project "/content/drive/MyDrive/Skripsi (Tugas Akhir)/Laboratory/UDAO-LASFNet/DroneVehicle" \
  --name "Ablation Study 1 (UDAO-Only)" \
  --save-period 20 \
  --exist-ok

!python val.py \
  --data data/dronevehicle_new.yaml \
  --weights "/content/drive/MyDrive/Skripsi (Tugas Akhir)/Laboratory/UDAO-LASFNet/DroneVehicle/Ablation Study 1 (UDAO-Only)/weights/best.pt" \
  --batch-size 32 \
  --imgsz 640 \
  --device 0 \
  --task val \
  --verbose

In [ ]:
# === ABLATION STUDY 2: DFATM-ONLY (Training + Evaluasi) ===
%cd /content/UDAO-LASFNet

!python train.py \
  --weights "" \
  --cfg models/lasfnet_dfatm_only.yaml \
  --data data/dronevehicle_new.yaml \
  --hyp data/hyps/hyp.scratch.yaml \
  --epochs 300 \
  --batch-size 48 \
  --imgsz 640 \
  --device 0 \
  --workers 8 \
  --cos-lr \
  --image-weights \
  --project "/content/drive/MyDrive/Skripsi (Tugas Akhir)/Laboratory/UDAO-LASFNet/DroneVehicle" \
  --name "Ablation Study 2 (DFATM-Only)" \
  --save-period 20 \
  --exist-ok

!python val.py \
  --data data/dronevehicle_new.yaml \
  --weights "/content/drive/MyDrive/Skripsi (Tugas Akhir)/Laboratory/UDAO-LASFNet/DroneVehicle/Ablation Study 2 (DFATM-Only)/weights/best.pt" \
  --batch-size 32 \
  --imgsz 640 \
  --device 0 \
  --task val \
  --verbose